In [ ]:
# | default_exp core

In [ ]:
# | export
from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from functools import lru_cache
import math
import pandas as pd
import duckdb
import structlog
from datetime import datetime
import json
import time

log = structlog.get_logger()

In [ ]:
# | export
IMPACT_PANELS = ("IMPACT341", "IMPACT410", "IMPACT468", "IMPACT505")
ACCESS_PANELS = ("ACCESS129", "ACCESS146")

VARIANT_KEY_COLS = [
    "Chromosome",
    "Start_Position",
    "End_Position",
    "Reference_Allele",
    "Tumor_Seq_Allele2",
]

MAF_USECOLS = [
    "Hugo_Symbol",
    "Tumor_Sample_Barcode",
    "Mutation_Status",
    "Variant_Classification",
    "Chromosome",
    "Start_Position",
    "End_Position",
    "Reference_Allele",
    "Tumor_Seq_Allele2",
    "t_ref_count",
    "t_alt_count",
]


@dataclass
class LabelConfig:
    """Configuration for the ctDNA labeling engine."""

    min_vaf: float = 0.01  # VAF threshold for Possible ctDNA+
    min_variants: int = 1  # Minimum # somatic SNVs passing VAF threshold
    min_fragments: int = 2000  # Min fragments for Depth QC
    access_panels: tuple[str, ...] = ACCESS_PANELS
    impact_panels: tuple[str, ...] = IMPACT_PANELS
    chunk_size: int = 15_000  # Metadata is ~1 row/sample; load in single batch
    ch_hotspot_maf: Path | None = None  # Optional CH hotspot MAF for variant demotion

    def __repr__(self) -> str:
        return (
            f"LabelConfig(min_vaf={self.min_vaf}, "
            f"min_variants={self.min_variants}, "
            f"chunk_size={self.chunk_size}, "
            f"ch_hotspot_maf={self.ch_hotspot_maf}, "
            f"panels={self.access_panels})"
        )


@dataclass
class Paths:
    """All input paths for the labeling pipeline."""

    cancer_samplesheet: Path
    healthy_xs1_samplesheet: Path
    healthy_xs2_samplesheet: Path
    cbioportal_dir: Path  # directory containing all 5 cBioPortal files
    krewlyzer_dirs: list[Path] = field(default_factory=list)

    def __post_init__(self):
        """Coerce all path fields from str to Path for safe / operations."""
        self.cancer_samplesheet = Path(self.cancer_samplesheet)
        self.healthy_xs1_samplesheet = Path(self.healthy_xs1_samplesheet)
        self.healthy_xs2_samplesheet = Path(self.healthy_xs2_samplesheet)
        self.cbioportal_dir = Path(self.cbioportal_dir)

        expanded = []
        for d in self.krewlyzer_dirs:
            p = Path(str(d).strip().strip('"').strip("'"))
            if p.is_file() and p.suffix == ".txt":
                with open(p) as f:
                    for line in f:
                        line = line.strip().strip('"').strip("'")
                        if line:
                            path_obj = Path(line)
                            if not path_obj.exists():
                                log.warning("manifest_path_missing", path=str(path_obj))
                            expanded.append(path_obj)
            else:
                if not p.exists():
                    log.warning("krewlyzer_dir_missing", path=str(p))
                expanded.append(p)
        self.krewlyzer_dirs = expanded

    @property
    def maf(self) -> Path:
        return self.cbioportal_dir / "data_mutations_extended.txt"

    @property
    def sv(self) -> Path:
        return self.cbioportal_dir / "data_sv.txt"

    @property
    def cna(self) -> Path:
        return self.cbioportal_dir / "data_CNA.txt"

    @property
    def clinical_sample(self) -> Path:
        return self.cbioportal_dir / "data_clinical_sample.txt"

    @property
    def clinical_patient(self) -> Path:
        return self.cbioportal_dir / "data_clinical_patient.txt"


In [ ]:
# | export
def load_samplesheet(path: str | Path) -> pd.DataFrame:
    """Load a krewlyzer samplesheet CSV."""
    if not Path(path).exists():
        log.error("samplesheet_missing", path=str(path))
        raise FileNotFoundError(f"Samplesheet not found: {path}")

    df = pd.read_csv(path)
    log.info("samplesheet_loaded", path=str(path), n_samples=len(df))
    return df


def get_sample_ids(samplesheet_path: str | Path) -> set[str]:
    """Extract unique sample IDs from a samplesheet."""
    df = load_samplesheet(samplesheet_path)
    if "sample" not in df.columns:
        log.error(
            "invalid_samplesheet", path=str(samplesheet_path), columns=list(df.columns)
        )
        raise KeyError("'sample' column missing in samplesheet")
    return set(df["sample"])


In [ ]:
# | export
@lru_cache(maxsize=1)
def load_maf(path: str | Path) -> pd.DataFrame:
    """Load MAF file with only the columns needed for labeling."""
    if not Path(path).exists():
        log.error("maf_missing", path=str(path))
        raise FileNotFoundError(f"MAF not found: {path}")

    log.info("loading_maf", path=str(path))
    try:
        df = pd.read_csv(
            path,
            sep="\t",
            comment="#",
            usecols=MAF_USECOLS,
            dtype={
                "t_ref_count": "Int64",
                "t_alt_count": "Int64",
                "Start_Position": "Int64",
                "End_Position": "Int64",
            },
        )
        log.info(
            "maf_loaded", n_rows=len(df), n_samples=df["Tumor_Sample_Barcode"].nunique()
        )
        return df
    except Exception as e:
        log.error("maf_load_failed", path=str(path), error=str(e))
        raise


@lru_cache(maxsize=1)
def load_sv(path: str | Path) -> pd.DataFrame:
    """Load structural variant file."""
    if not Path(path).exists():
        log.error("sv_missing", path=str(path))
        raise FileNotFoundError(f"SV file not found: {path}")

    log.info("loading_sv", path=str(path))
    try:
        df = pd.read_csv(path, sep="\t")
        log.info("sv_loaded", n_rows=len(df))
        return df
    except Exception as e:
        log.error("sv_load_failed", path=str(path), error=str(e))
        raise


@lru_cache(maxsize=1)
def load_cna(path: str | Path) -> pd.DataFrame:
    """Load discrete CNA matrix (wide format). Index = Hugo_Symbol."""
    if not Path(path).exists():
        log.error("cna_missing", path=str(path))
        raise FileNotFoundError(f"CNA not found: {path}")

    log.info("loading_cna", path=str(path))
    try:
        df = pd.read_csv(path, sep="\t", index_col=0)
        log.info("cna_loaded", n_genes=df.shape[0], n_samples=df.shape[1])
        return df
    except Exception as e:
        log.error("cna_load_failed", path=str(path), error=str(e))
        raise


@lru_cache(maxsize=1)
def load_clinical_sample(path: str | Path) -> pd.DataFrame:
    """Load sample-level clinical data (skips 4-line cBioPortal # header)."""
    if not Path(path).exists():
        log.error("clinical_sample_missing", path=str(path))
        raise FileNotFoundError(f"Clinical sample file not found: {path}")

    log.info("loading_clinical_sample", path=str(path))
    try:
        df = pd.read_csv(path, sep="\t", comment="#")
        log.info("clinical_sample_loaded", n_rows=len(df))
        return df
    except Exception as e:
        log.error("clinical_sample_load_failed", path=str(path), error=str(e))
        raise


@lru_cache(maxsize=1)
def load_clinical_patient(path: str | Path) -> pd.DataFrame:
    """Load patient-level clinical data (skips 4-line cBioPortal # header)."""
    if not Path(path).exists():
        log.error("clinical_patient_missing", path=str(path))
        raise FileNotFoundError(f"Clinical patient file not found: {path}")

    log.info("loading_clinical_patient", path=str(path))
    try:
        df = pd.read_csv(path, sep="\t", comment="#")
        log.info("clinical_patient_loaded", n_rows=len(df))
        return df
    except Exception as e:
        log.error("clinical_patient_load_failed", path=str(path), error=str(e))
        raise


# | export
def clear_cbioportal_caches():
    """Explicitly clear all cBioPortal LRU caches to prevent memory leaks in long running processes."""
    load_maf.cache_clear()
    load_clinical_sample.cache_clear()
    load_clinical_patient.cache_clear()
    log.info("cbioportal_caches_cleared")


In [ ]:
# | export
import threading

_thread_local = threading.local()


def get_duckdb_conn() -> duckdb.DuckDBPyConnection:
    """Create a thread-local DuckDB connection with optimal settings.
    Configured natively for 4 threads and 4GB memory to prevent HPC OOMs.
    """
    if not hasattr(_thread_local, "conn"):
        conn = duckdb.connect()
        conn.execute("SET threads TO 4")
        conn.execute("SET memory_limit = '4GB'")
        log.debug("duckdb_conn_initialized", threads=4, memory_limit="4GB")
        _thread_local.conn = conn
    return _thread_local.conn


def discover_available_samples(
    results_dirs: list[Path],
    required_suffix: str = ".metadata.parquet",
) -> dict[str, Path]:
    """Discover which samples have completed krewlyzer processing across multiple input directories."""
    available = {}
    for r_dir in results_dirs:
        results_path = Path(r_dir)
        if not results_path.exists():
            log.warning("results_dir_missing", path=str(results_path))
            continue

        # Auto-detect pipeline wrappers (Nextflow output dirs)
        scan_path = results_path
        if (results_path / "results").is_dir():
            scan_path = results_path / "results"

        for sample_dir in scan_path.iterdir():
            if sample_dir.is_dir():
                marker = sample_dir / f"{sample_dir.name}{required_suffix}"
                if marker.exists():
                    available[sample_dir.name] = scan_path

    log.info("sample_discovery", n_available=len(available), num_dirs=len(results_dirs))
    return available


In [ ]:
# | export
def _discover_feature_paths(
    feature_suffix: str,
    results_dirs: list[Path],
    sample_ids: list[str] | set[str] | None = None,
) -> list[str]:
    """Discover and return the list of parquet file paths for a feature type.

    This is a pure I/O function (filesystem ls) — no parquet reads occur.
    It is the shared first stage for both `load_feature_cohort` (batch) and
    `iter_feature_chunks` (streaming).

    Returns:
        Sorted list of absolute file path strings. Empty list if no files found.

    Raises:
        ValueError: If results_dirs is empty.
    """
    if not results_dirs:
        log.error("results_dirs_empty")
        raise ValueError("Must provide at least one krewlyzer results directory")

    # Step 1: Discover valid samples (fast filesystem ls, no parquet reads)
    log.info("discovering_samples", n_dirs=len(results_dirs))
    available_samples = discover_available_samples(
        results_dirs, required_suffix=".metadata.parquet"
    )

    if sample_ids is not None:
        final_samples = sorted(set(sample_ids).intersection(available_samples.keys()))
        if len(final_samples) < len(set(sample_ids)):
            log.warning(
                "samples_missing_krewlyzer_output",
                requested=len(set(sample_ids)),
                found=len(final_samples),
            )
    else:
        final_samples = sorted(available_samples.keys())

    if not final_samples:
        log.warning("no_samples_available_for_cohort", feature=feature_suffix)
        return []

    log.info("building_file_list", n_samples=len(final_samples))

    # Step 2: Build explicit file paths (no glob needed)
    file_paths = []
    missing_files = 0
    for sid in final_samples:
        r_dir = available_samples[sid]
        fp = r_dir / sid / f"{sid}{feature_suffix}"
        if fp.exists():
            file_paths.append(str(fp))
        else:
            missing_files += 1

    if missing_files > 0:
        log.info(
            "feature_files_missing",
            feature=feature_suffix,
            missing=missing_files,
            found=len(file_paths),
        )

    if not file_paths:
        log.warning("no_feature_files_found", feature=feature_suffix)

    return file_paths


def _read_parquet_chunk(
    conn: duckdb.DuckDBPyConnection,
    file_paths: list[str],
    feature_suffix: str,
    chunk_label: str = "",
    columns: list[str] | None = None,
) -> pd.DataFrame:
    """Read a batch of parquet files via DuckDB with retry logic.

    Args:
        conn: DuckDB connection (must be in the calling thread).
        file_paths: List of absolute parquet file paths to read.
        feature_suffix: Feature name for logging context.
        chunk_label: Human-readable label for log messages.
        columns: If set, SELECT only these columns instead of ``*``.
            Reduces memory for wide parquet files (e.g. WPSGenome).
            The ``sample_id`` column extracted from filename is always
            included regardless of this list.

    Returns:
        DataFrame with all rows from the batch, or empty DataFrame on
        permanent failure after 3 retries.
    """
    # Build column clause: either specific columns or SELECT *
    if columns:
        # Sanitize: only allow alphanumeric, underscore, dot (no SQL injection)
        safe_cols = [
            c for c in columns if all(ch.isalnum() or ch in ("_", ".") for ch in c)
        ]
        if not safe_cols:
            log.warning(
                "column_projection_empty_after_sanitize",
                feature=feature_suffix,
                requested=columns,
            )
            col_clause = "*"
        else:
            col_clause = ", ".join(safe_cols)
            log.debug(
                "column_projection_applied",
                feature=feature_suffix,
                n_columns=len(safe_cols),
                columns=safe_cols,
            )
    else:
        col_clause = "*"

    query = f"""
        SELECT {col_clause},
            regexp_extract(filename, '/([^/]+)/[^/]+$', 1) AS sample_id
        FROM read_parquet(
            ?,
            filename=true,
            union_by_name=true,
            hive_partitioning=false
        )
    """
    max_retries = 3
    for attempt in range(max_retries):
        try:
            return conn.execute(query, [file_paths]).df()
        except Exception as e:
            err_str = str(e)
            if attempt < max_retries - 1:
                log.warning(
                    "duckdb_io_retry",
                    feature=feature_suffix,
                    attempt=attempt + 1,
                    chunk=chunk_label,
                    error=err_str,
                )
                time.sleep(2**attempt)  # Exponential backoff for transient I/O
            else:
                log.error(
                    "duckdb_chunk_read_failed",
                    feature=feature_suffix,
                    chunk=chunk_label,
                    error=err_str,
                )
                return pd.DataFrame()
    return pd.DataFrame()


def _calculate_dynamic_chunk_size(
    conn: duckdb.DuckDBPyConnection,
    file_paths: list[str],
    target_rows: int = 15_000_000,
) -> int:
    """Probe the first 5 parquet files to estimate a safe chunk size.

    Uses ``COUNT(*)`` only — no data is loaded into Python memory.
    The resulting chunk size is clamped to ``[50, 15_000]`` so that:
    - Heavy features (FSC, ~30K rows/sample) → chunk ~500
    - Tiny features  (MDS, ~1 row/sample)   → chunk 15_000 (single sweep)

    Falls back to 500 if the probe fails (e.g. network I/O error).
    """
    if not file_paths:
        return 500

    probe = file_paths[: min(5, len(file_paths))]
    try:
        row = conn.execute(
            "SELECT count(*) FROM read_parquet(?, hive_partitioning=false)",
            [probe],
        ).fetchone()
        n_rows = row[0] if row else 0
        avg_rows = max(1, n_rows // len(probe))
        size = max(50, min(15_000, target_rows // avg_rows))
        log.info(
            "dynamic_chunk_size_calculated",
            probe_files=len(probe),
            total_probe_rows=n_rows,
            avg_rows_per_sample=avg_rows,
            chunk_size=size,
        )
        return size
    except Exception as exc:
        log.warning(
            "dynamic_chunk_probe_failed",
            error=str(exc),
            fallback=500,
        )
        return 500


def iter_feature_chunks(
    feature_suffix: str,
    results_dirs: list[Path],
    sample_ids: list[str] | set[str] | None = None,
    conn: duckdb.DuckDBPyConnection | None = None,
    chunk_size: int | str = "auto",
    columns: list[str] | None = None,
    target_rows: int = 15_000_000,
):
    """Stream feature data as chunks without accumulating in memory.

    This is the memory-safe alternative to `load_feature_cohort`. Each chunk
    contains complete samples (one parquet file = one sample), so chunk
    boundaries never split a sample's data.

    Args:
        feature_suffix: Parquet file suffix (e.g. '.FSC.ontarget.parquet').
        results_dirs: Krewlyzer output directories to scan.
        sample_ids: Optional subset of samples to include.
        conn: Optional DuckDB connection (created if not provided).
        chunk_size: Number of files per batch. 'auto' (default) probes the
            first 5 files to estimate rows-per-sample and self-tunes the
            batch size to target ``target_rows`` rows per chunk.
        columns: If set, DuckDB will SELECT only these columns instead
            of ``*``. Reduces memory for wide parquet files.
        target_rows: Target rows per chunk for auto-sizing (default 15M).
            Lower values reduce peak memory for heavy features.

    Yields:
        (chunk_df, chunk_idx, n_chunks): A tuple of the DataFrame chunk,
        the zero-based chunk index, and total number of chunks.

    Raises:
        ValueError: If results_dirs is empty.
    """
    file_paths = _discover_feature_paths(feature_suffix, results_dirs, sample_ids)
    if not file_paths:
        return  # Generator yields nothing — caller sees an empty iteration

    if conn is None:
        conn = get_duckdb_conn()

    conn.execute("SET threads=4;")  # Throttle SFTP I/O bursts

    # Resolve dynamic chunk sizing before entering the streaming loop
    if chunk_size == "auto":
        chunk_size = _calculate_dynamic_chunk_size(
            conn, file_paths, target_rows=target_rows
        )
    assert isinstance(chunk_size, int)

    n_chunks = math.ceil(len(file_paths) / chunk_size)
    log.info(
        "streaming_parquet_files",
        feature=feature_suffix,
        n_files=len(file_paths),
        chunk_size=chunk_size,
        n_chunks=n_chunks,
        column_projection=columns is not None,
        target_rows=target_rows,
    )

    for chunk_idx in range(n_chunks):
        start = chunk_idx * chunk_size
        batch = file_paths[start : start + chunk_size]
        chunk_label = f"{chunk_idx + 1}/{n_chunks}"

        df_chunk = _read_parquet_chunk(
            conn, batch, feature_suffix, chunk_label, columns=columns
        )

        if df_chunk.empty:
            log.warning(
                "empty_chunk",
                feature=feature_suffix,
                chunk=chunk_label,
                n_files=len(batch),
            )
            continue  # Skip empty chunks, don't abort the whole stream

        log.info(
            "chunk_loaded",
            feature=feature_suffix,
            chunk=chunk_label,
            n_samples=(
                df_chunk["sample_id"].nunique()
                if "sample_id" in df_chunk.columns
                else 0
            ),
            n_rows=len(df_chunk),
        )
        yield df_chunk, chunk_idx, n_chunks


def load_feature_cohort(
    feature_suffix: str,
    results_dirs: list[Path],
    sample_ids: list[str] | set[str] | None = None,
    conn: duckdb.DuckDBPyConnection | None = None,
    chunk_size: int | str = "auto",
) -> pd.DataFrame:
    """Load one feature type across available samples into a single DataFrame.

    This is the backward-compatible batch loader. For memory-constrained
    environments (e.g., HPC with 64GB RAM), use `iter_feature_chunks` instead
    to process data in a streaming fashion.

    Args:
        chunk_size: 'auto' (default) probes parquet row density at runtime.
            Pass an integer to override (e.g. 500 for large features).

    ARCHITECTURE NOTE: We build an explicit file list from discovered samples
    instead of using a glob pattern. This avoids DuckDB scanning thousands of
    directories over network mounts (SFTP/NFS), which causes multi-minute stalls.
    """
    start_time = time.time()

    chunks = []
    for df_chunk, _idx, _total in iter_feature_chunks(
        feature_suffix, results_dirs, sample_ids, conn, chunk_size
    ):
        chunks.append(df_chunk)

    df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

    elapsed = time.time() - start_time
    if df.empty:
        log.warning(
            "empty_result", feature=feature_suffix, reason="no_matching_rows_found"
        )
    else:
        log.info(
            "feature_cohort_loaded",
            feature=feature_suffix,
            n_samples=df["sample_id"].nunique(),
            n_rows=len(df),
            elapsed_sec=round(elapsed, 1),
        )
    return df


def load_metadata_cohort(
    results_dirs: list[Path], sample_ids=None, chunk_size: int | str = "auto"
):
    """Load metadata for all samples. Thin wrapper around load_feature_cohort.

    Metadata files are ~1 row per sample, so 'auto' will resolve to
    chunk_size=15_000 (single-sweep load) on first probe.
    """
    return load_feature_cohort(
        ".metadata.parquet", results_dirs, sample_ids, chunk_size=chunk_size
    )


def run_feature_sql(
    sql_query: str,
    feature_suffix: str,
    results_dirs: list[Path],
    sample_ids: list[str] | set[str] | None = None,
    conn: duckdb.DuckDBPyConnection | None = None,
) -> pd.DataFrame:
    """Execute a full-cohort SQL extraction in a single DuckDB pass.

    This is the SQL pushdown alternative to ``iter_feature_chunks`` +
    per-sample ``extract()``. It runs the evaluator's SQL query against
    all discovered parquet files at once.

    Most evaluators return one row per sample. Some may return a "tall"
    result with multiple rows per sample (e.g. one row per region_type);
    the CLI handles pivoting to wide format if needed.

    The query must contain a ``read_parquet(?, ...)`` placeholder that
    accepts the file path list.

    Args:
        sql_query: DuckDB SQL with ``?`` placeholder for file paths.
        feature_suffix: Parquet suffix (e.g. '.MDS.ontarget.parquet').
        results_dirs: Krewlyzer output directories to scan.
        sample_ids: Optional subset to filter to.
        conn: Optional DuckDB connection.

    Returns:
        DataFrame with extraction results, or empty DataFrame on failure.
    """
    file_paths = _discover_feature_paths(feature_suffix, results_dirs, sample_ids)
    if not file_paths:
        log.warning("run_feature_sql_no_files", feature=feature_suffix)
        return pd.DataFrame()

    if conn is None:
        conn = get_duckdb_conn()

    start = time.time()
    max_retries = 3
    for attempt in range(max_retries):
        try:
            df = conn.execute(sql_query, [file_paths]).df()
            elapsed = time.time() - start
            log.info(
                "sql_pushdown_complete",
                feature=feature_suffix,
                n_files=len(file_paths),
                n_rows=len(df),
                n_samples=(
                    df["sample_id"].nunique() if "sample_id" in df.columns else len(df)
                ),
                elapsed_sec=round(elapsed, 1),
            )
            return df
        except Exception as exc:
            if attempt < max_retries - 1:
                log.warning(
                    "sql_pushdown_retry",
                    feature=feature_suffix,
                    attempt=attempt + 1,
                    error=str(exc),
                )
                time.sleep(2**attempt)
            else:
                log.error(
                    "sql_pushdown_failed",
                    feature=feature_suffix,
                    error=str(exc),
                )
                return pd.DataFrame()
    return pd.DataFrame()


In [ ]:
# | export
def load_sample_feature(
    sample_id: str,
    feature_suffix: str,
    results_dir: str | Path,
) -> pd.DataFrame:
    """Load a single parquet feature file for one sample. (Useful for single-sample logic)"""
    path = Path(results_dir) / sample_id / f"{sample_id}{feature_suffix}"
    if not path.exists():
        log.warning("file_missing", path=str(path), sample_id=sample_id)
        raise FileNotFoundError(f"Feature file not found: {path}")

    try:
        return pd.read_parquet(path)
    except Exception as e:
        log.error("single_feature_load_failed", path=str(path), error=str(e))
        raise


def load_sample_metadata(
    sample_id: str,
    results_dir: str | Path,
) -> dict:
    """Load the metadata.parquet for a sample -> dict of QC values."""
    try:
        df = load_sample_feature(sample_id, ".metadata.parquet", results_dir)
        if df.empty:
            log.warning("empty_metadata", sample_id=sample_id)
            return {}
        return df.iloc[0].to_dict()
    except Exception as e:
        log.error("metadata_fetch_failed", sample_id=sample_id, error=str(e))
        return {}


In [ ]:
# | export
def make_variant_key(df: pd.DataFrame) -> pd.Series:
    """Create a hashable variant key from genomic coordinates."""
    if df.empty:
        log.warning("empty_df_in_variant_key")
        return pd.Series(dtype=object)

    for col in VARIANT_KEY_COLS:
        if col not in df.columns:
            log.error("missing_variant_key_col", col=col)
            raise KeyError(f"Missing required col for variant key: {col}")

    return df[VARIANT_KEY_COLS].apply(
        lambda row: (
            row["Chromosome"],
            int(row["Start_Position"]),
            int(row["End_Position"]),
            row["Reference_Allele"],
            row["Tumor_Seq_Allele2"],
        ),
        axis=1,
    )


@dataclass
class EvalRun:
    """Records the exact conditions of an evaluation run."""

    run_id: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    n_total_labeled: int = 0
    n_evaluated_samples: int = 0
    features_evaluated: list[str] = field(default_factory=list)
    label_config: LabelConfig = field(default_factory=LabelConfig)

    def to_json(self, path: str | Path):
        try:
            with open(path, "w") as f:
                data = self.__dict__.copy()
                data["label_config"] = self.label_config.__dict__
                json.dump(data, f, indent=2)
            log.info("eval_run_saved", path=str(path), run_id=self.run_id)
        except Exception as e:
            log.error("eval_run_save_failed", path=str(path), error=str(e))
            raise


In [ ]:
#| export
# Metadata columns that every *_matrix.parquet carries from the labeling step.
# These are NOT features — they must be deduplicated, not prefixed, during fusion.
LABEL_META_COLS = {
    "SAMPLE_ID",
    "PATIENT_ID",
    "CANCER_TYPE",
    "CANCER_TYPE_DETAILED",
    "ONCOTREE_CODE",
    "SAMPLE_TYPE",
    "GENE_PANEL",
    "label",
    "split",  # train/test/exclude — assigned by label pipeline (v0.0.16+)
    "has_impact_match",
    "has_snv",
    "has_sv",
    "has_cna",
    "has_paired_impact",
    "max_vaf",
    "mean_vaf",
    "std_vaf",
    "n_impact_confirmed",
    "n_somatic_snvs",
    "n_total_somatic_snvs",
    "n_somatic_svs",
    "n_cna_events",
    "n_ch_variants",
    "n_non_ch_variants",
    "access_version",
    "min_vaf_used",
    "min_variants_used",
    "total_fragments_pf",
    "sample_id",
    "filename",
    "assay",
}


def fuse_matrices(
    output_dir: str | Path,
    *,
    min_evaluators: int = 1,
    drop_low_variance: bool = True,
    variance_threshold: float = 0.0,
    output_name: str = "super_matrix.parquet",
) -> pd.DataFrame:
    """Discover per-evaluator matrices and fuse into a single super-matrix.

    Each ``*_matrix.parquet`` file in ``output_dir`` contains label/metadata
    columns plus evaluator-specific feature columns. This function:

    1. Discovers all ``*_matrix.parquet`` files (skips ``super_matrix.parquet``
       and ``scoreboard_*`` files).
    2. Extracts metadata columns once from the first matrix.
    3. Prefixes each evaluator's feature columns with the evaluator name
       to prevent collisions (e.g., ``FSCOnTarget__ratio_short``).
    4. Outer-joins all feature DataFrames on ``SAMPLE_ID``.
    5. Filters to samples appearing in at least ``min_evaluators`` matrices.
    6. Drops low-variance features (cross-evaluator QC) when enabled.
    7. Writes the result to ``output_dir / output_name``.

    Args:
        output_dir: Directory containing ``*_matrix.parquet`` files.
        min_evaluators: Minimum number of evaluators a sample must appear in.
        drop_low_variance: If True, drop features with variance at or below
            ``variance_threshold`` after fusion. Eliminates constant columns
            that carry no discriminative signal (step 3A.3 of the plan).
        variance_threshold: Features with variance <= this value are dropped.
            Default 0.0 drops only exactly-constant features.
        output_name: Filename for the output parquet.

    Returns:
        The fused DataFrame.
    """
    import glob

    out_path = Path(output_dir)
    matrix_paths = sorted(glob.glob(str(out_path / "*_matrix.parquet")))

    # Exclude the output itself and scoreboard files to avoid recursion
    matrix_paths = [
        p
        for p in matrix_paths
        if not Path(p).name.startswith("super_matrix")
        and not Path(p).name.startswith("scoreboard_")
    ]

    if not matrix_paths:
        log.warning("fuse_no_matrices", output_dir=str(out_path))
        return pd.DataFrame()

    log.info("fuse_started", n_matrices=len(matrix_paths), output_dir=str(out_path))

    metadata_df = None
    feature_dfs = []
    evaluator_names = []

    for matrix_path in matrix_paths:
        eval_name = Path(matrix_path).name.replace("_matrix.parquet", "")
        evaluator_names.append(eval_name)

        try:
            df = pd.read_parquet(matrix_path)
        except Exception as exc:
            log.error(
                "fuse_read_failed",
                evaluator=eval_name,
                path=matrix_path,
                error=str(exc),
            )
            continue

        if "SAMPLE_ID" not in df.columns:
            log.error("fuse_missing_sample_id", evaluator=eval_name, path=matrix_path)
            continue

        # Separate metadata from feature columns
        present_meta = [c for c in df.columns if c in LABEL_META_COLS]
        feature_cols = [c for c in df.columns if c not in LABEL_META_COLS]

        # Capture metadata from the first valid matrix
        if metadata_df is None and present_meta:
            metadata_df = df[present_meta].copy()
            log.info(
                "fuse_metadata_captured",
                evaluator=eval_name,
                n_samples=len(metadata_df),
                n_meta_cols=len(present_meta),
            )

        if not feature_cols:
            log.warning("fuse_no_features", evaluator=eval_name)
            continue

        # Prefix feature columns with evaluator name to prevent collisions
        feat_df = df[["SAMPLE_ID"] + feature_cols].copy()
        feat_df = feat_df.rename(columns={c: f"{eval_name}__{c}" for c in feature_cols})

        feature_dfs.append(feat_df)
        log.info(
            "fuse_evaluator_loaded",
            evaluator=eval_name,
            n_samples=len(feat_df),
            n_features=len(feature_cols),
        )

    if not feature_dfs:
        log.error("fuse_no_valid_matrices", output_dir=str(out_path))
        return pd.DataFrame()

    # Outer-join all feature DataFrames on SAMPLE_ID
    fused = feature_dfs[0]
    for feat_df in feature_dfs[1:]:
        fused = pd.merge(fused, feat_df, on="SAMPLE_ID", how="outer")

    # Count how many evaluators each sample appears in
    eval_presence = pd.DataFrame({"SAMPLE_ID": fused["SAMPLE_ID"]})
    for i, feat_df in enumerate(feature_dfs):
        eval_presence[f"_eval_{i}"] = eval_presence["SAMPLE_ID"].isin(
            feat_df["SAMPLE_ID"]
        )
    fused["n_evaluators"] = eval_presence.iloc[:, 1:].sum(axis=1).astype(int)

    # Filter by min_evaluators
    before_filter = len(fused)
    fused = fused[fused["n_evaluators"] >= min_evaluators].copy()
    n_dropped = before_filter - len(fused)
    if n_dropped > 0:
        log.info(
            "fuse_min_evaluator_filter",
            min_evaluators=min_evaluators,
            n_dropped=n_dropped,
            n_remaining=len(fused),
        )

    # Re-attach metadata via left join
    if metadata_df is not None:
        # Deduplicate metadata (same sample may appear in multiple matrices)
        metadata_df = metadata_df.drop_duplicates(subset=["SAMPLE_ID"])
        fused = pd.merge(metadata_df, fused, on="SAMPLE_ID", how="right")

    # ── Cross-evaluator feature selection (Plan step 3A.3) ──
    # Remove features with near-zero variance that can't contribute to
    # discrimination. This runs AFTER fusion across all evaluators — a
    # global QC step rather than per-evaluator.
    if drop_low_variance:
        feature_cols = [c for c in fused.columns if "__" in c]
        numeric_feats = fused[feature_cols].select_dtypes(include="number")
        variances = numeric_feats.var()
        low_var_cols = variances[variances <= variance_threshold].index.tolist()
        if low_var_cols:
            fused = fused.drop(columns=low_var_cols)
            log.info(
                "fuse_low_variance_dropped",
                n_dropped=len(low_var_cols),
                threshold=variance_threshold,
                examples=low_var_cols[:5],
            )

    # Write output
    out_file = out_path / output_name
    fused.to_parquet(out_file, index=False)

    n_features = len([c for c in fused.columns if "__" in c])
    log.info(
        "fuse_complete",
        n_samples=len(fused),
        n_total_cols=len(fused.columns),
        n_feature_cols=n_features,
        n_evaluators_fused=len(feature_dfs),
        evaluators=evaluator_names,
        output=str(out_file),
    )

    return fused